In [1]:
print("H")

H


In [2]:
# precompute.ipynb
# Generates all JSON files that the backend reads at startup.
# Run this AFTER exp1.2 and exp1.3 have finished and saved their .pt weight files.
#
# Files produced (copy these to backend/precomputed/ and backend/weights/):
#   bitcoin_stats.json           <- GET /api/bitcoin/stats
#   bitcoin_timeseries.json      <- GET /api/bitcoin/timeseries
#   bitcoin_predictions.json     <- used by /transaction, /graph, /clusters endpoints
#   bitcoin_clusters.json        <- GET /api/bitcoin/clusters
#   bitcoin_feature_importance.json  <- top-10 features per node for /transaction

import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd
import numpy as np
import networkx as nx
import json

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GATConv

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

c:\Users\OM MAHAJAN\miniconda3\envs\isro\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [4]:
# Load raw data — same pipeline as exp1.2 / exp1.3
DATA = '../../data/elliptic_bitcoin_dataset/'

features = pd.read_csv(DATA + 'elliptic_txs_features.csv', header=None)
classes  = pd.read_csv(DATA + 'elliptic_txs_classes.csv')
edgelist = pd.read_csv(DATA + 'elliptic_txs_edgelist.csv')

features.columns = (
    ['txId', 'time_step']
    + [f'local_feat_{i}' for i in range(93)]
    + [f'agg_feat_{i}'   for i in range(72)]
)
features['txId'] = features['txId'].astype(str)

# Raw CSV: '1' = illicit, '2' = licit, 'unknown' = unknown
classes.columns = ['txId', 'class']
classes['txId']  = classes['txId'].astype(str)
classes['class'] = classes['class'].astype(str).map(
    {'1': 'illicit', '2': 'licit', 'unknown': 'unknown'}
)

edgelist['txId1'] = edgelist['txId1'].astype(str)
edgelist['txId2'] = edgelist['txId2'].astype(str)

df = features.merge(classes, on='txId', how='left')
df['class'] = df['class'].fillna('unknown')
df = df.reset_index(drop=True)

print('Total nodes :', len(df))
print('Total edges :', len(edgelist))
print(df['class'].value_counts())

Total nodes : 203769
Total edges : 234355
class
unknown    157205
licit       42019
illicit      4545
Name: count, dtype: int64


In [5]:
# -------------------------------------------------------------------
# SECTION 1: bitcoin_stats.json
# Simple dataset-level counts, returned by GET /api/bitcoin/stats
# -------------------------------------------------------------------

illicit_count = (df['class'] == 'illicit').sum()
licit_count   = (df['class'] == 'licit').sum()
unknown_count = (df['class'] == 'unknown').sum()
labelled      = illicit_count + licit_count

stats = {
    'total_nodes'        : int(len(df)),
    'total_edges'        : int(len(edgelist)),
    'illicit_count'      : int(illicit_count),
    'licit_count'        : int(licit_count),
    'unknown_count'      : int(unknown_count),
    'labelled_count'     : int(labelled),
    'illicit_ratio'      : round(illicit_count / labelled, 4),
    'time_steps'         : int(df['time_step'].nunique()),
    'time_step_min'      : int(df['time_step'].min()),
    'time_step_max'      : int(df['time_step'].max()),
    'avg_degree'         : round(
        (2 * len(edgelist)) / len(df), 4
    ),  # undirected average degree
}

with open('bitcoin_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('bitcoin_stats.json written')
print(json.dumps(stats, indent=2))

bitcoin_stats.json written
{
  "total_nodes": 203769,
  "total_edges": 234355,
  "illicit_count": 4545,
  "licit_count": 42019,
  "unknown_count": 157205,
  "labelled_count": 46564,
  "illicit_ratio": 0.0976,
  "time_steps": 49,
  "time_step_min": 1,
  "time_step_max": 49,
  "avg_degree": 2.3002
}


In [6]:
# -------------------------------------------------------------------
# SECTION 2: bitcoin_timeseries.json
# Illicit / licit / unknown counts per time step
# Returned by GET /api/bitcoin/timeseries
# -------------------------------------------------------------------

ts_rows = []
for step in sorted(df['time_step'].unique()):
    sub = df[df['time_step'] == step]
    ts_rows.append({
        'time_step' : int(step),
        'total'     : int(len(sub)),
        'illicit'   : int((sub['class'] == 'illicit').sum()),
        'licit'     : int((sub['class'] == 'licit').sum()),
        'unknown'   : int((sub['class'] == 'unknown').sum()),
    })

with open('bitcoin_timeseries.json', 'w') as f:
    json.dump(ts_rows, f, indent=2)

print(f'bitcoin_timeseries.json written ({len(ts_rows)} time steps)')
print('Preview (first 3):', ts_rows[:3])

bitcoin_timeseries.json written (49 time steps)
Preview (first 3): [{'time_step': 1, 'total': 7880, 'illicit': 17, 'licit': 2130, 'unknown': 5733}, {'time_step': 2, 'total': 4544, 'illicit': 18, 'licit': 1099, 'unknown': 3427}, {'time_step': 3, 'total': 6621, 'illicit': 11, 'licit': 1268, 'unknown': 5342}]


In [11]:
# -------------------------------------------------------------------
# SECTION 3: Build the PyG graph (shared by predictions + importance)
# Identical to exp1.2 / exp1.3 setup
# -------------------------------------------------------------------

tx_to_idx = {tx: idx for idx, tx in enumerate(df['txId'])}

src = edgelist['txId1'].map(tx_to_idx)
dst = edgelist['txId2'].map(tx_to_idx)
mask = src.notna() & dst.notna()
src = src[mask].astype(int).values
dst = dst[mask].astype(int).values

edge_index = torch.tensor(
    np.stack([np.concatenate([src, dst]),
              np.concatenate([dst, src])], axis=0),
    dtype=torch.long
)

FEATURE_COLS = (
    [f'local_feat_{i}' for i in range(93)]
    + [f'agg_feat_{i}'   for i in range(72)]
)
x_np = df[FEATURE_COLS].values.astype(np.float32)
x_np = (x_np - x_np.mean(0, keepdims=True)) / (x_np.std(0, keepdims=True) + 1e-8)

x = torch.tensor(x_np, dtype=torch.float)

data = Data(x=x, edge_index=edge_index)
data = data.to(DEVICE)

print('PyG graph ready. x:', x.shape, '  edge_index:', edge_index.shape)

PyG graph ready. x: torch.Size([203769, 165])   edge_index: torch.Size([2, 468710])


In [15]:
# -------------------------------------------------------------------
# SECTION 4: Load trained models
# Weights must already exist (produced by exp1.2 and exp1.3)
# -------------------------------------------------------------------

# GraphSAGE — matches models/bitcoin/graphsage.py
class GraphSAGEModel(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=2):
        super().__init__()
        self.conv1      = SAGEConv(in_channels, hidden_channels)
        self.conv2      = SAGEConv(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.dropout    = nn.Dropout(p=0.3)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        return self.classifier(x)


# GAT — matches models/bitcoin/gat.py
class GATModel(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=2, heads=4):
        super().__init__()
        self.conv1      = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.3)
        self.conv2      = GATConv(hidden_channels * heads, hidden_channels,
                                  heads=1, concat=False, dropout=0.3)
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))
        return self.classifier(x)


graphsage = GraphSAGEModel(in_channels=data.x.shape[1], hidden_channels=64, out_channels=2).to(DEVICE)
graphsage.load_state_dict(torch.load('graphsage_best.pt', map_location=DEVICE))
graphsage.eval()

gat = GATModel(in_channels=data.x.shape[1], hidden_channels=64, out_channels=2, heads=4).to(DEVICE)
gat.load_state_dict(torch.load('gat_best.pt', map_location=DEVICE))
gat.eval()

print('GraphSAGE and GAT weights loaded.')

GraphSAGE and GAT weights loaded.


In [16]:
# -------------------------------------------------------------------
# SECTION 5: bitcoin_predictions.json
# Full-graph inference on all nodes using both models.
# Format expected by the backend (from precompute_predictions.py outline):
# {
#   "txId": {
#     "graphsage_risk_score": 0.87,
#     "gat_risk_score": 0.91,
#     "risk_score": 0.87,         <- graphsage is the default model
#     "predicted_class": "illicit"
#   }, ...
# }
# -------------------------------------------------------------------

with torch.no_grad():
    sage_logits = graphsage(data.x, data.edge_index)          # [N, 2]
    sage_proba  = F.softmax(sage_logits, dim=1)[:, 1].cpu().numpy()  # P(illicit)

    gat_logits  = gat(data.x, data.edge_index)
    gat_proba   = F.softmax(gat_logits, dim=1)[:, 1].cpu().numpy()

print('Inference done. Building predictions dict...')

predictions = {}
for idx, row in df.iterrows():
    tx_id  = row['txId']
    sg_score = round(float(sage_proba[idx]), 4)
    gt_score = round(float(gat_proba[idx]),  4)
    # default model = graphsage (matches backend config: GRAPHSAGE_WEIGHTS is primary)
    pred_class = 'illicit' if sg_score >= 0.5 else 'licit'

    predictions[tx_id] = {
        'graphsage_risk_score': sg_score,
        'gat_risk_score'      : gt_score,
        'risk_score'          : sg_score,
        'predicted_class'     : pred_class,
        'true_class'          : row['class'],
    }

with open('bitcoin_predictions.json', 'w') as f:
    json.dump(predictions, f)

print(f'bitcoin_predictions.json written ({len(predictions)} entries)')

# Quick sanity check on labelled nodes
labelled_preds  = {k: v for k, v in predictions.items() if v['true_class'] != 'unknown'}
correct = sum(
    1 for v in labelled_preds.values()
    if v['predicted_class'] == v['true_class']
)
print(f'Accuracy on labelled nodes (GraphSAGE default): {correct / len(labelled_preds):.4f}')

Inference done. Building predictions dict...
bitcoin_predictions.json written (203769 entries)
Accuracy on labelled nodes (GraphSAGE default): 0.9715


In [17]:
# -------------------------------------------------------------------
# SECTION 6: bitcoin_clusters.json
# Uses NetworkX weakly connected components on the directed edgelist.
# Filters by illicit_ratio >= threshold OR avg_risk_score >= threshold.
# Matches cluster_detector.py in the backend spec.
# -------------------------------------------------------------------

MIN_CLUSTER_SIZE           = 3
ILLICIT_RATIO_THRESHOLD    = 0.20
AVG_RISK_THRESHOLD         = 0.60

print('Building NetworkX graph for cluster detection...')
G = nx.from_pandas_edgelist(
    edgelist, source='txId1', target='txId2',
    create_using=nx.DiGraph()
)

# Create a fast lookup: txId -> class
class_lookup = dict(zip(df['txId'], df['class']))
tstep_lookup = dict(zip(df['txId'], df['time_step']))

print(f'Running weakly connected components on {G.number_of_nodes()} nodes...')
clusters = []
for i, component in enumerate(nx.weakly_connected_components(G)):
    if len(component) < MIN_CLUSTER_SIZE:
        continue

    node_list = list(component)

    # Count illicit nodes
    classes_in_cluster = [class_lookup.get(tx, 'unknown') for tx in node_list]
    illicit_count = classes_in_cluster.count('illicit')
    illicit_ratio = illicit_count / len(node_list)

    # Average risk score from predictions
    risk_scores = [
        predictions[tx]['risk_score']
        for tx in node_list if tx in predictions
    ]
    avg_risk = sum(risk_scores) / len(risk_scores) if risk_scores else 0.0

    # Only keep suspicious clusters
    if illicit_ratio < ILLICIT_RATIO_THRESHOLD and avg_risk < AVG_RISK_THRESHOLD:
        continue

    time_steps = [int(tstep_lookup.get(tx, 0)) for tx in node_list if tx in tstep_lookup]

    clusters.append({
        'cluster_id'   : f'CLR-{i:04d}',
        'node_ids'     : node_list,
        'node_count'   : len(node_list),
        'illicit_count': illicit_count,
        'illicit_ratio': round(illicit_ratio, 4),
        'avg_risk_score': round(avg_risk, 4),
        'time_step_min': int(min(time_steps)) if time_steps else 0,
        'time_step_max': int(max(time_steps)) if time_steps else 0,
    })

# Sort by avg risk descending
clusters.sort(key=lambda c: c['avg_risk_score'], reverse=True)

with open('bitcoin_clusters.json', 'w') as f:
    json.dump(clusters, f, indent=2)

print(f'bitcoin_clusters.json written ({len(clusters)} suspicious clusters)')
if clusters:
    top = clusters[0]
    print(f'Top cluster: {top["cluster_id"]}  nodes={top["node_count"]}  illicit_ratio={top["illicit_ratio"]}  avg_risk={top["avg_risk_score"]}')

Building NetworkX graph for cluster detection...
Running weakly connected components on 203769 nodes...
bitcoin_clusters.json written (0 suspicious clusters)


In [18]:
# -------------------------------------------------------------------
# SECTION 7: bitcoin_feature_importance.json
# Gradient x Input attribution for each labelled node.
# For each node we compute: abs(gradient_of_illicit_logit * input_feature)
# Then pick the top-10 features by importance.
#
# Only done for labelled nodes (not unknown) to keep the file size manageable.
# The backend uses GraphSAGE as the default model for feature attribution.
# -------------------------------------------------------------------

labelled_mask = df['class'] != 'unknown'
labelled_indices = df.index[labelled_mask].tolist()

print(f'Computing gradient x input for {len(labelled_indices)} labelled nodes...')
print('This may take a few minutes on CPU.')

# We need gradients w.r.t. x, so enable grad for x
x_attr = data.x.clone().detach().requires_grad_(True)

# Single forward pass — gradients flow through the entire graph
graphsage.eval()
logits_attr = graphsage(x_attr, data.edge_index)   # [N, 2]

# Sum illicit logits for ALL labelled nodes — then backprop once
# This gives grad for each node simultaneously (efficient single backward pass)
labelled_idx_tensor = torch.tensor(labelled_indices, dtype=torch.long)
target_logits = logits_attr[labelled_idx_tensor, 1]   # illicit logit for labelled nodes
target_logits.sum().backward()                         # one backward pass

# Gradient tensor [N, 166]
grads = x_attr.grad.detach().cpu().numpy()   # shape [N, 166]
x_vals = data.x.detach().cpu().numpy()       # shape [N, 166]

# Gradient x Input importance: abs(grad * feature_value)
importance_mat = np.abs(grads * x_vals)      # [N, 166]

print('Building feature importance dict...')

feature_importance = {}
for node_idx in labelled_indices:
    tx_id       = df.loc[node_idx, 'txId']
    importances = importance_mat[node_idx]   # [166]
    feat_values = x_vals[node_idx]           # [166]

    # Get top-10 feature indices by importance
    top10_idx = np.argsort(importances)[::-1][:10]

    top10 = [
        {
            'name'      : FEATURE_COLS[fi],
            'importance': round(float(importances[fi]), 6),
            'value'     : round(float(feat_values[fi]),  6),
        }
        for fi in top10_idx
    ]
    feature_importance[tx_id] = top10

with open('bitcoin_feature_importance.json', 'w') as f:
    json.dump(feature_importance, f)

print(f'bitcoin_feature_importance.json written ({len(feature_importance)} nodes)')

# Show top features for one example node
sample_id = df.loc[labelled_indices[0], 'txId']
print(f'\nExample top-3 features for txId={sample_id}:')
for feat in feature_importance[sample_id][:3]:
    print(f"  {feat['name']}: importance={feat['importance']}, value={feat['value']}")

Computing gradient x input for 46564 labelled nodes...
This may take a few minutes on CPU.
Building feature importance dict...
bitcoin_feature_importance.json written (46564 nodes)

Example top-3 features for txId=232438397:
  local_feat_6: importance=52.895035, value=12.414557
  local_feat_75: importance=43.773357, value=4.313295
  local_feat_3: importance=43.583946, value=12.409294


In [19]:
# -------------------------------------------------------------------
# FINAL: Summary of all generated files
# -------------------------------------------------------------------
import os

OUTPUT_FILES = [
    'bitcoin_stats.json',
    'bitcoin_timeseries.json',
    'bitcoin_predictions.json',
    'bitcoin_clusters.json',
    'bitcoin_feature_importance.json',
    'bitcoin_metrics.json',      # written by exp1.3
    'graphsage_best.pt',         # written by exp1.2
    'gat_best.pt',               # written by exp1.3
]

print('=' * 55)
print('PRECOMPUTE COMPLETE — file sizes:')
print('=' * 55)
for fname in OUTPUT_FILES:
    if os.path.exists(fname):
        size_kb = os.path.getsize(fname) / 1024
        print(f'  {fname:<45} {size_kb:>8.1f} KB')
    else:
        print(f'  {fname:<45}  MISSING (run the corresponding notebook first)')

print()
print('Next steps:')
print('  1. Copy bitcoin_stats.json, bitcoin_timeseries.json,')
print('     bitcoin_predictions.json, bitcoin_clusters.json,')
print('     bitcoin_feature_importance.json, bitcoin_metrics.json')
print('     -> backend/precomputed/')
print()
print('  2. Copy graphsage_best.pt, gat_best.pt')
print('     -> backend/weights/')
print()
print('  3. uvicorn main:app --reload   (from the backend/ directory)')

PRECOMPUTE COMPLETE — file sizes:
  bitcoin_stats.json                                 0.3 KB
  bitcoin_timeseries.json                            5.4 KB
  bitcoin_predictions.json                       28859.3 KB
  bitcoin_clusters.json                              0.0 KB
  bitcoin_feature_importance.json                32159.9 KB
  bitcoin_metrics.json                               0.3 KB
  graphsage_best.pt                                119.3 KB
  gat_best.pt                                      237.4 KB

Next steps:
  1. Copy bitcoin_stats.json, bitcoin_timeseries.json,
     bitcoin_predictions.json, bitcoin_clusters.json,
     bitcoin_feature_importance.json, bitcoin_metrics.json
     -> backend/precomputed/

  2. Copy graphsage_best.pt, gat_best.pt
     -> backend/weights/

  3. uvicorn main:app --reload   (from the backend/ directory)
